**SignalState For LSA6**

read JSON
reconstruct secondwise states
identify valid programs
infer code meanings
generate compact programs
map signal groups to SUMO tls indices
assign morning/evening scenario program


In [1]:
from pathlib import Path
import json
import pandas as pd

# ============================================================
# SETTINGS
# ============================================================

IN_DIR = Path(
    r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\signal_states\LD-LSA16_6c5e44f1-3aae-4c29-926c-8db5b61da400"
)

OUT_DIR = Path(
    r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\workshop"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Start with only one signal group
SIGNAL_ID = 2          # LSA16 West, K2
APPROACH = "West"

# choose one hour
WINDOW_START = "08:00:00"
WINDOW_END   = "09:00:00"

# ============================================================
# PARSE ONE SIGNAL ID
# ============================================================

rows = []

for path in sorted(IN_DIR.glob("*.json")):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    start_utc = pd.to_datetime(data["start"])

    for value in data.get("values", []):
        timestamp_utc = start_utc + pd.to_timedelta(value.get("offset", 0), unit="ms")
        timestamp_local = timestamp_utc.tz_convert("Europe/Berlin")

        sig_states = {
            item["id"]: item["sgState"]
            for item in value.get("sigState", [])
        }

        if SIGNAL_ID in sig_states:
            node_info = value.get("nodes", [{}])[0]

            rows.append({
                "file": path.name,
                "timestamp_utc": timestamp_utc,
                "timestamp_local": timestamp_local,
                "date": timestamp_local.date(),
                "time": timestamp_local.time(),
                "approach": APPROACH,
                "signal_id": SIGNAL_ID,
                "sg_state": sig_states[SIGNAL_ID],
                "program": node_info.get("program"),
                "stage": node_info.get("stage"),
                "cycCnt": node_info.get("cycCnt"),
                "stgCnt": node_info.get("stgCnt"),
                "stgTran": node_info.get("stgTran"),
            })

df = pd.DataFrame(rows)

# Filter selected one-hour window
start_t = pd.to_datetime(WINDOW_START).time()
end_t = pd.to_datetime(WINDOW_END).time()

df_hour = df[
    (df["time"] >= start_t) &
    (df["time"] < end_t)
].copy()

# Save secondwise data
out_secondwise = OUT_DIR / f"LSA16_{APPROACH}_signalID{SIGNAL_ID}_{WINDOW_START[:2]}-{WINDOW_END[:2]}_secondwise.csv"
df_hour.to_csv(out_secondwise, index=False, encoding="utf-8-sig")

print("Saved secondwise file:")
print(out_secondwise)

# ============================================================
# SUMMARY: how long each sgState appears
# ============================================================

state_summary = (
    df_hour
    .groupby(["date", "sg_state"])
    .size()
    .reset_index(name="duration_seconds")
    .sort_values(["date", "sg_state"])
)

out_summary = OUT_DIR / f"LSA16_{APPROACH}_signalID{SIGNAL_ID}_{WINDOW_START[:2]}-{WINDOW_END[:2]}_sgState_duration_summary.csv"
state_summary.to_csv(out_summary, index=False, encoding="utf-8-sig")

print("Saved sgState duration summary:")
print(out_summary)

print(state_summary.head(50))

Saved secondwise file:
C:\Users\mogul\OneDrive\Masaüstü\mt_emre\workshop\LSA16_West_signalID2_08-09_secondwise.csv
Saved sgState duration summary:
C:\Users\mogul\OneDrive\Masaüstü\mt_emre\workshop\LSA16_West_signalID2_08-09_sgState_duration_summary.csv
          date  sg_state  duration_seconds
0   2026-03-01         2              2774
1   2026-03-01         3               102
2   2026-03-01         4               690
3   2026-03-01         5                34
4   2026-03-02         2              2432
5   2026-03-02         3               120
6   2026-03-02         4              1008
7   2026-03-02         5                40
8   2026-03-03         2              2373
9   2026-03-03         3               120
10  2026-03-03         4              1067
11  2026-03-03         5                40
12  2026-03-04         2              2438
13  2026-03-04         3               120
14  2026-03-04         4              1002
15  2026-03-04         5                40
16  2026-03-05  

**only for representative day and observed time window**

In [3]:
from pathlib import Path
import json
import pandas as pd
from openpyxl.styles import PatternFill, Font, Border, Side, Alignment
from openpyxl.utils import get_column_letter

IN_DIR = Path(
    r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\signal_states\LD-LSA16_6c5e44f1-3aae-4c29-926c-8db5b61da400"
)

OUT_DIR = Path(
    r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\workshop"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_XLSX = OUT_DIR / "LSA16_selected_days_signal_state_duration_colored.xlsx"

APPROACH_SIGNAL_MAP = {
    "West":  {"signal_group": "K2", "responsible_detector": "WD21 : id 3", "signal_id": 2},
    "South": {"signal_group": "K1", "responsible_detector": "VD11 : id 1", "signal_id": 1},
    "East":  {"signal_group": "K4", "responsible_detector": "WD41 : id 8", "signal_id": 4},
    "North": {"signal_group": "K3", "responsible_detector": "VD32 : id 7", "signal_id": 3},
}

TARGET_SCENARIOS = {
    "morning_2026-03-10_0800_0900": {
        "date": "2026-03-10",
        "start": "08:00:00",
        "end": "09:00:00",
    },
    "evening_2026-03-23_1600_1700": {
        "date": "2026-03-23",
        "start": "16:00:00",
        "end": "17:00:00",
    },
}

STATE_MEANING = {
    4: "green",
    2: "red_or_blocked",
    3: "transition",
    5: "transition",
}

STATE_COLOR = {
    "green": "C6EFCE",
    "red_or_blocked": "FFC7CE",
    "transition": "FFEB9C",
    "unknown": "D9EAD3",
}

rows = []

for path in sorted(IN_DIR.glob("*.json")):
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        start_utc = pd.to_datetime(data["start"])

        for value in data.get("values", []):
            timestamp_utc = start_utc + pd.to_timedelta(value.get("offset", 0), unit="ms")
            timestamp_local = timestamp_utc.tz_convert("Europe/Berlin")

            sig_states = {
                item["id"]: item["sgState"]
                for item in value.get("sigState", [])
            }

            node_info = value.get("nodes", [{}])[0]

            for approach, info in APPROACH_SIGNAL_MAP.items():
                signal_id = info["signal_id"]

                if signal_id in sig_states:
                    sg_state = sig_states[signal_id]

                    rows.append({
                        "file": path.name,
                        "timestamp_utc": timestamp_utc,
                        "timestamp_local": timestamp_local,
                        "date": str(timestamp_local.date()),
                        "time": timestamp_local.time(),
                        "approach": approach,
                        "signal_group": info["signal_group"],
                        "responsible_detector": info["responsible_detector"],
                        "signal_id": signal_id,
                        "sg_state": sg_state,
                        "state_meaning": STATE_MEANING.get(sg_state, "unknown"),
                        "program": node_info.get("program"),
                        "stage": node_info.get("stage"),
                        "cycCnt": node_info.get("cycCnt"),
                        "stgCnt": node_info.get("stgCnt"),
                        "stgTran": node_info.get("stgTran"),
                    })

    except Exception as e:
        print(f"Could not parse {path.name}: {e}")

df = pd.DataFrame(rows)

if df.empty:
    raise ValueError("No data was parsed. Check the input folder path.")

filtered_parts = []

for scenario_name, scenario in TARGET_SCENARIOS.items():
    start_t = pd.to_datetime(scenario["start"]).time()
    end_t = pd.to_datetime(scenario["end"]).time()

    temp = df[
        (df["date"] == scenario["date"]) &
        (df["time"] >= start_t) &
        (df["time"] < end_t)
    ].copy()

    temp["scenario"] = scenario_name
    filtered_parts.append(temp)

df_selected = pd.concat(filtered_parts, ignore_index=True)

if df_selected.empty:
    raise ValueError("No rows found for selected dates and time windows.")

summary = (
    df_selected
    .groupby([
        "scenario",
        "date",
        "approach",
        "signal_group",
        "responsible_detector",
        "signal_id",
        "sg_state",
        "state_meaning"
    ])
    .size()
    .reset_index(name="duration_seconds")
    .sort_values(["scenario", "approach", "sg_state"])
)

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="All_Approaches", index=False)

    for approach in ["West", "South", "East", "North"]:
        approach_df = summary[summary["approach"] == approach].copy()
        approach_df.to_excel(writer, sheet_name=approach, index=False)

    df_selected.to_excel(writer, sheet_name="Secondwise_Data", index=False)

wb = pd.ExcelFile(OUT_XLSX)
from openpyxl import load_workbook

workbook = load_workbook(OUT_XLSX)

header_fill = PatternFill("solid", fgColor="D9EAF7")
header_font = Font(bold=True)
thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)

for ws in workbook.worksheets:
    ws.freeze_panes = "A2"

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.border = thin_border
        cell.alignment = Alignment(horizontal="center", vertical="center")

    headers = [cell.value for cell in ws[1]]
    state_meaning_col = headers.index("state_meaning") + 1 if "state_meaning" in headers else None

    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        state_meaning = None

        if state_meaning_col:
            state_meaning = row[state_meaning_col - 1].value

        fill_color = STATE_COLOR.get(state_meaning, "FFFFFF")
        fill = PatternFill("solid", fgColor=fill_color)

        for cell in row:
            cell.fill = fill
            cell.border = thin_border
            cell.alignment = Alignment(horizontal="center", vertical="center")

    for col_idx, column_cells in enumerate(ws.columns, start=1):
        max_length = 0
        for cell in column_cells:
            if cell.value is not None:
                max_length = max(max_length, len(str(cell.value)))
        ws.column_dimensions[get_column_letter(col_idx)].width = min(max_length + 3, 38)

workbook.save(OUT_XLSX)

print("Saved colored Excel file:")
print(OUT_XLSX)

print("\nSummary preview:")
print(summary)

ValueError: Excel does not support datetimes with timezones. Please ensure that datetimes are timezone unaware before writing to Excel.